# GeoThought Baseline Evaluation

Evaluates four autoregressive baselines on **two datasets** — GeoThought val split and
Geometry3K test set — so results are directly comparable to LatentEuclid v4 (45.38% SOTA).

| # | Model | Input | Generation |
|---|-------|-------|------------|
| 1 | `Qwen3-VL-4B-Instruct` direct | image + question | greedy, 50 tokens |
| 2 | `Qwen3-4B-Base` text-only | question only | greedy, 50 tokens |
| 3 | `Qwen3-4B-Base` reasoning | question only | sampled, 1024 tokens |
| 4 | `Qwen3-VL-4B-Instruct` reasoning | image + question | sampled, 1024 tokens |

**Parsing parity**: all models go through the same `extract_and_clean()` pipeline as `eval_e2e.py`.

**Output**: per-dataset JSON files saved to `RESULTS_DIR/{geothought,geometry3k}/`.

In [2]:
# ── Cell 1: Environment ───────────────────────────────────────────────────────
import sys, os

REPO_PATH    = "/home/ec2-user/work/MMML-Project"
RESULTS_DIR  = "/home/ec2-user/work/MMML-Project/data/baselines"
DATA_CACHE   = "/home/ec2-user/work/MMML-Project/data/geothought_hf"
G3K_PROMPTS  = "/home/ec2-user/work/MMML-Project/GeoThought/evaluation_script/geometry3k_test_prompts.jsonl"
G3K_IMG_ROOT = "/home/ec2-user/work/MMML-Project/GeoThought/evaluation_script"

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
os.makedirs(os.path.join(RESULTS_DIR, "geothought"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "geometry3k"), exist_ok=True)

print(f"Repo:       {REPO_PATH}")
print(f"Results:    {RESULTS_DIR}/{{geothought,geometry3k}}/")


Repo:       /home/ec2-user/work/MMML-Project
Results:    /home/ec2-user/work/MMML-Project/data/baselines/{geothought,geometry3k}/


In [3]:
# ── Cell 2: Device detection ──────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DEVICE = "cpu"
    print("No GPU found — falling back to CPU")

import transformers as _tf
print(f"transformers: {_tf.__version__}  |  torch: {torch.__version__}  |  Device: {DEVICE}")

CUDA GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM:     102.0 GB
transformers: 5.5.4  |  torch: 2.7.0+cu128  |  Device: cuda


In [4]:
# ── Cell 3: Datasets + splits ────────────────────────────────────────────────
import json, re, random
from PIL import Image
from datasets import load_dataset, load_from_disk

JSONL_PATH = os.path.join(REPO_PATH, "data/geothoughts_verified.jsonl")
GT_PATH    = os.path.join(REPO_PATH, "data/ground_truths.json")

if os.path.exists(DATA_CACHE):
    print("Loading HF dataset from local cache...")
    hf_dataset = load_from_disk(DATA_CACHE)
else:
    print("Downloading from HuggingFace (one-time, will cache to disk)...")
    hf_dataset = load_dataset("xinlingdedeng/Geo-Thought", split="train")
    hf_dataset.save_to_disk(DATA_CACHE)
print(f"HF dataset size: {len(hf_dataset)}")

with open(JSONL_PATH) as f:
    full_data = [json.loads(line) for line in f]
with open(GT_PATH) as f:
    ground_truths = json.load(f)

# ── GeoThought val split (must match eval_e2e.py exactly) ────────────────────
random.seed(42)
random.shuffle(full_data)
split_idx = int(0.9 * len(full_data))
val_data  = full_data[split_idx:]

def _load_hf_image(item):
    m = re.search(r'sample_(\d+)', item.get("image_path", ""))
    if m:
        hf_row = hf_dataset[int(m.group(1))]
        img = hf_row.get("image") or hf_row.get("images")
        if img is not None:
            return img.convert("RGB") if hasattr(img, "convert") else Image.fromarray(img).convert("RGB")
    return Image.new("RGB", (224, 224), (128, 128, 128))

geothought_items = []
for item in val_data:
    gt = ground_truths.get(item["image_path"])
    if gt is None:
        continue
    geothought_items.append({
        "image_path": item["image_path"],
        "question":   item["question"],
        "gt_raw":     str(gt),
        "image":      _load_hf_image(item),
    })
print(f"GeoThought val:   {len(geothought_items)} samples")

# ── Geometry3K test set ───────────────────────────────────────────────────────
with open(G3K_PROMPTS) as f:
    g3k_raw = [json.loads(line) for line in f]

geometry3k_items = []
for item in g3k_raw:
    img_path = os.path.join(G3K_IMG_ROOT, item["image_path"])
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception:
        img = Image.new("RGB", (224, 224), (128, 128, 128))
    geometry3k_items.append({
        "image_path": item["image_path"],
        "question":   item["question"],
        "gt_raw":     str(item["ground_truth"]),
        "image":      img,
    })
print(f"Geometry3K test:  {len(geometry3k_items)} samples")

# Shared alias used by all eval cells — swap to run on the other dataset
val_items = geothought_items


Loading HF dataset from local cache...
HF dataset size: 17077
GeoThought val:   595 samples
Geometry3K test:  601 samples


In [8]:
# ── Cell 4: Evaluation utilities ──────────────────────────────────────────────
# Inlined from data/evaluate_generated.py to avoid PYTHONPATH dependencies.
# Functions are UNCHANGED — identical to what eval_e2e.py imports — so parsing
# is bit-for-bit equivalent across all baselines and LatentEuclid.
import math as _math
import re as _re

def clean_base_model_ans(ans):
    """From data/evaluate_generated.py — unchanged."""
    ans = ans.strip()
    ans = _re.sub(r'[*]*angstroms.*', '', ans, flags=_re.DOTALL)
    ans = _re.sub(r'[*]*wsj.*', '', ans, flags=_re.DOTALL)
    ans = _re.split(r'[\n\u4e00-\u9fff\u3040-\u30ff]', ans)[0]
    match = _re.match(r'^([-+]?[\d\.\/\s\*\+\-\(\)pisqrt]+)', ans)
    if match:
        ans = match.group(1)
    return ans.strip()

def safe_math_eval(expr_str):
    """From data/evaluate_generated.py — unchanged."""
    expr_str = _re.sub(r'(\d)(pi|sqrt)', r'\1*\2', str(expr_str))
    expr_str = _re.sub(r'\)(pi|sqrt|\d)', r')*\1', expr_str)
    expr_str = _re.sub(r'(pi)(\d)', r'\1*\2', expr_str)
    expr_str = expr_str.replace(')(', ')*(')     
    allowed_names = {"pi": _math.pi, "sqrt": _math.sqrt, "__builtins__": {}}
    safe_chars = set("0123456789.+-*/()pisqrt ")
    if not all(c in safe_chars for c in expr_str):
        return None
    import ast, warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            ast.parse(expr_str)
            val = eval(expr_str, allowed_names)
            return float(val) if val is not None else None
        except Exception:
            return None

def normalize(ans):
    """From data/evaluate_generated.py — unchanged."""
    ans = str(ans).strip()
    ans = _re.sub(r'(?:cm|mm|km|m|meters?|inches?|feet|foot|ft|nautical\smiles?|nauticalmiles?|nautical|miles?|yards?|yd|units?|square|sq|s|in)\b',
                  '', ans, flags=_re.IGNORECASE)
    ans = ans.replace('^2', '').replace('^', '').replace(' ', '')
    ans = ans.replace('(', '').replace(')', '').replace('[', '').replace(']', '')
    ans = ans.replace(':', '/')
    f_val = safe_math_eval(ans)
    if f_val is not None:
        return str(int(f_val)) if f_val == int(f_val) else str(round(f_val, 4))
    return ans.lower()

def extract_and_clean(generation: str) -> str:
    """
    Unified answer extractor used identically for all baselines.
    Mirrors y_decoder_prefix.py's generate() which splits on 'Answer:',
    then eval_e2e.py's clean_base_model_ans call.
    """
    if "Answer:" in generation:
        generation = generation.split("Answer:")[-1]
    return clean_base_model_ans(generation)

def score(pred_raw: str, gt_raw: str):
    """Returns (is_correct, gt_norm, pred_norm) — same logic as eval_e2e.py."""
    gt_norm   = normalize(gt_raw)
    pred_norm = normalize(pred_raw)
    is_correct = (gt_norm == pred_norm)
    if not is_correct:
        gt_val   = safe_math_eval(gt_raw)
        pred_val = safe_math_eval(pred_raw)
        if gt_val is not None and pred_val is not None:
            is_correct = _math.isclose(gt_val, pred_val, rel_tol=1e-3, abs_tol=0.06)
    return is_correct, gt_norm, pred_norm

print("Evaluation utilities ready.")

Evaluation utilities ready.


In [5]:
# ── Cell 5: Shared evaluation runner ─────────────────────────────────────────
import gc
from tqdm import tqdm

def run_evaluation(items, generate_fn, model_name, batch_size, results_subdir, filename):
    """
    Generic eval loop. Saves to RESULTS_DIR/{results_subdir}/{filename}.
    Returns (results_list, summary_dict).
    """
    output_path = os.path.join(RESULTS_DIR, results_subdir, filename)
    results = []
    correct = 0

    for i in tqdm(range(0, len(items), batch_size), desc=f"{model_name} [{results_subdir}]"):
        batch  = items[i : i + batch_size]
        images = [item["image"]    for item in batch]
        qs     = [item["question"] for item in batch]

        generations = generate_fn(images, qs)

        for item, gen in zip(batch, generations):
            pred_raw = extract_and_clean(gen)
            is_ok, gt_norm, pred_norm = score(pred_raw, item["gt_raw"])
            if is_ok:
                correct += 1
            results.append({
                "image":            item["image_path"],
                "is_correct":       is_ok,
                "gt_raw":           item["gt_raw"],
                "pred_raw":         pred_raw,
                "gt_norm":          gt_norm,
                "pred_norm":        pred_norm,
                "model_generation": gen.strip(),
            })

        if (i // batch_size + 1) % 10 == 0:
            with open(output_path, "w") as f:
                json.dump(results, f, indent=2)

    total = len(results)
    acc   = correct / total * 100 if total > 0 else 0.0
    summary = {"model": model_name, "dataset": results_subdir,
               "correct": correct, "total": total, "accuracy": acc}

    with open(output_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n{'='*55}")
    print(f"{model_name}  [{results_subdir}]")
    print(f"Correct: {correct}/{total}  |  Accuracy: {acc:.2f}%")
    print(f"Saved → {output_path}")
    print('='*55)
    return results, summary

print("Runner ready.")


Runner ready.


## Models 1 & 3 — `Qwen3-VL-4B-Instruct`

Both variants share the same model weights; only the system prompt and generation parameters differ.
Load once, run both, then unload before loading the base model.

- **Direct**: concise system prompt, greedy decoding, 50 new tokens.
- **Reasoning**: step-by-step system prompt, sampled decoding (`temperature=0.7`), 1 024 new tokens.
  Sampling is necessary to activate the model's chain-of-thought mode.

Both use `<image>` stripped from the question text (the image is provided via the content array).

In [9]:
# ── Cell 6: Load Qwen3-VL-4B-Instruct ────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForImageTextToText

VL_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

print(f"Loading {VL_MODEL_ID}...")

# Qwen3-VL requires transformers >= 4.52. If AutoProcessor returns a bare
# Qwen2TokenizerFast instead of a VL processor, images will be silently
# ignored and the VL baseline will be meaningless. Detect and warn.
vl_processor = AutoProcessor.from_pretrained(VL_MODEL_ID, trust_remote_code=True)

_proc_type = type(vl_processor).__name__
print(f"Processor type: {_proc_type}")

_is_vl_processor = hasattr(vl_processor, "image_processor")
if not _is_vl_processor:
    import transformers as _tf
    print(f"\n⚠  WARNING: AutoProcessor returned '{_proc_type}' (no image_processor).")
    print(f"   transformers version: {_tf.__version__}")
    print("   Qwen3-VL requires transformers >= 4.52.")
    print("   Image inputs will NOT be encoded — VL results will be text-only.\n")

# AutoProcessor for Qwen3-VL may return a composite VL processor (with .tokenizer
# attribute) or a plain tokenizer directly (old transformers). Handle both.
_vl_tok = getattr(vl_processor, "tokenizer", vl_processor)
_vl_tok.padding_side = "left"   # required for batched left-padded generation
if _vl_tok.pad_token is None:
    _vl_tok.pad_token = _vl_tok.eos_token

vl_model = AutoModelForImageTextToText.from_pretrained(
    VL_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map={"" : DEVICE},
)
vl_model.eval()
params_b = sum(p.numel() for p in vl_model.parameters()) / 1e9
print(f"Model loaded ({params_b:.2f}B params). VL processor: {_is_vl_processor}")

Loading Qwen/Qwen3-VL-4B-Instruct...
Processor type: Qwen3VLProcessor


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Model loaded (4.44B params). VL processor: True


In [10]:
# ── Cell 7: Model 1 — VL direct (both datasets) ─────────────────────────────
VL_DIRECT_BATCH   = 32
VL_DIRECT_MAX_TOK = 50
_SYSTEM_DIRECT = "You are a concise math assistant. Answer geometry questions with the numerical value only. No explanation, no units."

def _vl_build_inputs(images, questions, system_prompt):
    messages_batch = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text":  q.replace("<image>", "").strip() + "\nAnswer:"},
            ]},
        ]
        for img, q in zip(images, questions)
    ]
    texts = [
        vl_processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]
    return vl_processor(
        text=texts, images=images, padding=True, return_tensors="pt"
    ).to(DEVICE)

def vl_direct_generate(images, questions):
    inputs = _vl_build_inputs(images, questions, _SYSTEM_DIRECT)
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out_ids = vl_model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=VL_DIRECT_MAX_TOK,
                repetition_penalty=1.05,
                pad_token_id=_vl_tok.pad_token_id,
            )
    input_len  = inputs["input_ids"].shape[-1]
    new_tokens = out_ids[:, input_len:]
    return _vl_tok.batch_decode(new_tokens, skip_special_tokens=True)

results_vl_direct_gt, summary_vl_direct_gt = run_evaluation(
    geothought_items, vl_direct_generate,
    "Qwen3-VL-4B-Instruct (direct)", VL_DIRECT_BATCH,
    "geothought", "eval_qwen_vl_direct.json",
)
results_vl_direct_g3k, summary_vl_direct_g3k = run_evaluation(
    geometry3k_items, vl_direct_generate,
    "Qwen3-VL-4B-Instruct (direct)", VL_DIRECT_BATCH,
    "geometry3k", "eval_qwen_vl_direct.json",
)


Qwen3-VL-4B-Instruct (direct) [geothought]: 100%|██████████| 19/19 [00:29<00:00,  1.55s/it]



Qwen3-VL-4B-Instruct (direct)  [geothought]
Correct: 219/595  |  Accuracy: 36.81%
Saved → /home/ec2-user/work/MMML-Project/data/baselines/geothought/eval_qwen_vl_direct.json


Qwen3-VL-4B-Instruct (direct) [geometry3k]: 100%|██████████| 19/19 [00:41<00:00,  2.18s/it]


Qwen3-VL-4B-Instruct (direct)  [geometry3k]
Correct: 107/601  |  Accuracy: 17.80%
Saved → /home/ec2-user/work/MMML-Project/data/baselines/geometry3k/eval_qwen_vl_direct.json


In [11]:
# ── Cell 8: Model 4 — VL reasoning (both datasets) ──────────────────────────
VL_REASONING_BATCH   = 8
VL_REASONING_MAX_TOK = 1024
_SYSTEM_REASONING = (
    "You are a geometry expert. Think step by step before answering. "
    "Show your reasoning, then end with exactly: Answer: [number]"
)

def vl_reasoning_generate(images, questions):
    inputs = _vl_build_inputs(images, questions, _SYSTEM_REASONING)
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out_ids = vl_model.generate(
                **inputs,
                do_sample=True,
                temperature=0.7,
                top_p=0.8,
                top_k=20,
                max_new_tokens=VL_REASONING_MAX_TOK,
                repetition_penalty=1.05,
                pad_token_id=_vl_tok.pad_token_id,
            )
    input_len  = inputs["input_ids"].shape[-1]
    new_tokens = out_ids[:, input_len:]
    return _vl_tok.batch_decode(new_tokens, skip_special_tokens=True)

results_vl_reasoning_gt, summary_vl_reasoning_gt = run_evaluation(
    geothought_items, vl_reasoning_generate,
    "Qwen3-VL-4B-Instruct (reasoning)", VL_REASONING_BATCH,
    "geothought", "eval_qwen_vl_reasoning.json",
)
results_vl_reasoning_g3k, summary_vl_reasoning_g3k = run_evaluation(
    geometry3k_items, vl_reasoning_generate,
    "Qwen3-VL-4B-Instruct (reasoning)", VL_REASONING_BATCH,
    "geometry3k", "eval_qwen_vl_reasoning.json",
)


Qwen3-VL-4B-Instruct (reasoning) [geothought]: 100%|██████████| 75/75 [56:04<00:00, 44.86s/it]



Qwen3-VL-4B-Instruct (reasoning)  [geothought]
Correct: 307/595  |  Accuracy: 51.60%
Saved → /home/ec2-user/work/MMML-Project/data/baselines/geothought/eval_qwen_vl_reasoning.json


Qwen3-VL-4B-Instruct (reasoning) [geometry3k]: 100%|██████████| 76/76 [59:33<00:00, 47.03s/it] 


Qwen3-VL-4B-Instruct (reasoning)  [geometry3k]
Correct: 239/601  |  Accuracy: 39.77%
Saved → /home/ec2-user/work/MMML-Project/data/baselines/geometry3k/eval_qwen_vl_reasoning.json


In [12]:
# ── Cell 9: Unload VL model ───────────────────────────────────────────────────
del vl_model, vl_processor
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free after VL model unload: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

VRAM free after VL model unload: 20.8 GB


## Model 2 — `Qwen3-4B-Base` text-only

Uses the **same base model** as the LatentEuclid Y-Decoder, with no image input.
This isolates the contribution of visual grounding: if LatentEuclid beats this baseline,
the latent thought vectors carry information the question text alone cannot provide.

Prompt format: `"{question}\nAnswer:"` — identical continuation style to the Y-Decoder's
`generate()` method (`text_prompts = [q.strip() + "\nAnswer: " for q in questions]`).
The `<image>` placeholder is stripped since no visual input is given.

In [13]:
# ── Cell 10: Load Qwen3-4B-Base ───────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = "Qwen/Qwen3-4B-Base"

print(f"Loading {BASE_MODEL_ID} on {DEVICE}...")
print("First run will download ~8 GB — subsequent runs use the local HF cache.")

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_tokenizer.padding_side = "left"
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map={"": DEVICE},
)
base_model.eval()
print(f"Model loaded ({sum(p.numel() for p in base_model.parameters()) / 1e9:.2f}B params).")

Loading Qwen/Qwen3-4B-Base on cuda...
First run will download ~8 GB — subsequent runs use the local HF cache.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded (4.02B params).


In [14]:
# ── Cell 11: Model 2 — Base text-only (both datasets) ───────────────────────
BASE_BATCH   = 128
BASE_MAX_TOK = 50

def base_text_generate(images, questions):
    prompts = [q.replace("<image>", "").strip() + "\nAnswer:" for q in questions]
    inputs = base_tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(DEVICE)
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out_ids = base_model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=BASE_MAX_TOK,
                repetition_penalty=1.1,
                eos_token_id=[base_tokenizer.eos_token_id, 151645, 151643],
                pad_token_id=base_tokenizer.pad_token_id,
            )
    input_len  = inputs["input_ids"].shape[-1]
    new_tokens = out_ids[:, input_len:]
    return base_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

results_base_text_gt, summary_base_text_gt = run_evaluation(
    geothought_items, base_text_generate,
    "Qwen3-4B-Base (text-only)", BASE_BATCH,
    "geothought", "eval_qwen_base_text.json",
)
results_base_text_g3k, summary_base_text_g3k = run_evaluation(
    geometry3k_items, base_text_generate,
    "Qwen3-4B-Base (text-only)", BASE_BATCH,
    "geometry3k", "eval_qwen_base_text.json",
)


Qwen3-4B-Base (text-only) [geothought]: 100%|██████████| 5/5 [00:21<00:00,  4.39s/it]



Qwen3-4B-Base (text-only)  [geothought]
Correct: 5/595  |  Accuracy: 0.84%
Saved → /home/ec2-user/work/MMML-Project/data/baselines/geothought/eval_qwen_base_text.json


Qwen3-4B-Base (text-only) [geometry3k]: 100%|██████████| 5/5 [00:20<00:00,  4.03s/it]


Qwen3-4B-Base (text-only)  [geometry3k]
Correct: 24/601  |  Accuracy: 3.99%
Saved → /home/ec2-user/work/MMML-Project/data/baselines/geometry3k/eval_qwen_base_text.json


## Model 4 — `Qwen3-4B-Base` reasoning (text-only CoT)

Same base model as Model 2 and the LatentEuclid Y-Decoder, but prompted to reason
step by step before answering. Completion-style prompt (no system message) to suit
the base model's training distribution.

This is the **fair AR CoT baseline**: it shows how much chain-of-thought helps the
base model when there is no image, and gives an upper bound on what the Y-Decoder
could achieve if it could reason freely in text rather than reading latent prefixes.

In [15]:
# ── Cell 12: Model 3 — Base reasoning (both datasets) ───────────────────────
BASE_REASONING_BATCH   = 32
BASE_REASONING_MAX_TOK = 1024

def base_reasoning_generate(images, questions):
    prompts = [
        q.replace("<image>", "").strip() +
        "\nSolution: Let me solve this step by step.\nAnswer:"
        for q in questions
    ]
    inputs = base_tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(DEVICE)
    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out_ids = base_model.generate(
                **inputs,
                do_sample=True,
                temperature=0.7,
                top_p=0.8,
                top_k=20,
                max_new_tokens=BASE_REASONING_MAX_TOK,
                repetition_penalty=1.1,
                eos_token_id=[base_tokenizer.eos_token_id, 151645, 151643],
                pad_token_id=base_tokenizer.pad_token_id,
            )
    input_len  = inputs["input_ids"].shape[-1]
    new_tokens = out_ids[:, input_len:]
    return base_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

results_base_reasoning_gt, summary_base_reasoning_gt = run_evaluation(
    geothought_items, base_reasoning_generate,
    "Qwen3-4B-Base (reasoning)", BASE_REASONING_BATCH,
    "geothought", "eval_qwen_base_reasoning.json",
)
results_base_reasoning_g3k, summary_base_reasoning_g3k = run_evaluation(
    geometry3k_items, base_reasoning_generate,
    "Qwen3-4B-Base (reasoning)", BASE_REASONING_BATCH,
    "geometry3k", "eval_qwen_base_reasoning.json",
)


Qwen3-4B-Base (reasoning) [geothought]: 100%|██████████| 19/19 [24:02<00:00, 75.94s/it]



Qwen3-4B-Base (reasoning)  [geothought]
Correct: 2/595  |  Accuracy: 0.34%
Saved → /home/ec2-user/work/MMML-Project/data/baselines/geothought/eval_qwen_base_reasoning.json


Qwen3-4B-Base (reasoning) [geometry3k]: 100%|██████████| 19/19 [22:29<00:00, 71.02s/it]


Qwen3-4B-Base (reasoning)  [geometry3k]
Correct: 10/601  |  Accuracy: 1.66%
Saved → /home/ec2-user/work/MMML-Project/data/baselines/geometry3k/eval_qwen_base_reasoning.json


In [ ]:
# ── Cell 12: Unload base model ────────────────────────────────────────────────
del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free after base model unload: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

## Results — Summary & Combined Output

In [ ]:
# ── Cell 13: Combine results + save ──────────────────────────────────────────
def _load_results(subdir, filename):
    p = os.path.join(RESULTS_DIR, subdir, filename)
    return json.load(open(p)) if os.path.exists(p) else []

def _index(results): return {r["image"]: r for r in results}

# Load all results for both datasets
results_vl_direct_gt       = _load_results("geothought", "eval_qwen_vl_direct.json")
results_vl_direct_g3k      = _load_results("geometry3k",  "eval_qwen_vl_direct.json")
results_vl_reasoning_gt    = _load_results("geothought", "eval_qwen_vl_reasoning.json")
results_vl_reasoning_g3k   = _load_results("geometry3k",  "eval_qwen_vl_reasoning.json")
results_base_text_gt       = _load_results("geothought", "eval_qwen_base_text.json")
results_base_text_g3k      = _load_results("geometry3k",  "eval_qwen_base_text.json")
results_base_reasoning_gt  = _load_results("geothought", "eval_qwen_base_reasoning.json")
results_base_reasoning_g3k = _load_results("geometry3k",  "eval_qwen_base_reasoning.json")

def _combined(items, *idx_key_pairs):
    rows = []
    for item in items:
        k = item["image_path"]
        row = {"image": k, "question": item["question"],
               "gt_raw": item["gt_raw"], "gt_norm": normalize(item["gt_raw"])}
        for idx, col in idx_key_pairs:
            d = idx.get(k)
            row[col] = {"model_generation": d["model_generation"],
                        "pred_raw": d["pred_raw"], "pred_norm": d["pred_norm"],
                        "is_correct": d["is_correct"]} if d else None
        rows.append(row)
    return rows

combined_gt = _combined(
    geothought_items,
    (_index(results_vl_direct_gt),      "qwen_vl_direct"),
    (_index(results_base_text_gt),      "qwen_base_text"),
    (_index(results_base_reasoning_gt), "qwen_base_reasoning"),
    (_index(results_vl_reasoning_gt),   "qwen_vl_reasoning"),
)
combined_g3k = _combined(
    geometry3k_items,
    (_index(results_vl_direct_g3k),      "qwen_vl_direct"),
    (_index(results_base_text_g3k),      "qwen_base_text"),
    (_index(results_base_reasoning_g3k), "qwen_base_reasoning"),
    (_index(results_vl_reasoning_g3k),   "qwen_vl_reasoning"),
)

json.dump(combined_gt,  open(os.path.join(RESULTS_DIR, "geothought", "eval_baseline_combined.json"), "w"), indent=2)
json.dump(combined_g3k, open(os.path.join(RESULTS_DIR, "geometry3k",  "eval_baseline_combined.json"), "w"), indent=2)
print(f"Combined saved: geothought ({len(combined_gt)}) | geometry3k ({len(combined_g3k)})")

# ── Summary table ─────────────────────────────────────────────────────────────
LATENT_EUCLID_V4 = {"model": "LatentEuclid v4 (SOTA)", "correct": None, "total": 595, "accuracy": 45.38}

def _make_summaries(res_vl_d, res_vl_r, res_base_t, res_base_r):
    rows = []
    for name, res in [
        ("Qwen3-VL-4B-Instruct (direct)",    res_vl_d),
        ("Qwen3-4B-Base (text-only)",         res_base_t),
        ("Qwen3-4B-Base (reasoning)",         res_base_r),
        ("Qwen3-VL-4B-Instruct (reasoning)",  res_vl_r),
    ]:
        if not res: continue
        c = sum(1 for r in res if r["is_correct"])
        rows.append({"model": name, "correct": c, "total": len(res),
                     "accuracy": c / len(res) * 100})
    return rows

summaries_gt  = _make_summaries(results_vl_direct_gt,  results_vl_reasoning_gt,
                                 results_base_text_gt,   results_base_reasoning_gt)
summaries_g3k = _make_summaries(results_vl_direct_g3k, results_vl_reasoning_g3k,
                                 results_base_text_g3k,  results_base_reasoning_g3k)

json.dump(summaries_gt  + [LATENT_EUCLID_V4],
          open(os.path.join(RESULTS_DIR, "geothought", "eval_baseline_summary.json"), "w"), indent=2)
json.dump(summaries_g3k,
          open(os.path.join(RESULTS_DIR, "geometry3k",  "eval_baseline_summary.json"), "w"), indent=2)

def _print_table(summaries, title, latent_row=None):
    print(f"\n{'='*62}  {title}")
    print(f"{'Model':<44} {'Correct':>8} {'Total':>6} {'Acc':>7}")
    print("-"*62)
    for s in summaries:
        c = str(s['correct']) if s['correct'] is not None else '—'
        print(f"{s['model']:<44} {c:>8} {s['total']:>6} {s['accuracy']:>6.2f}%")
    if latent_row:
        print(f"{latent_row['model']:<44} {'—':>8} {latent_row['total']:>6} {latent_row['accuracy']:>6.2f}%")
    print('='*62)

_print_table(summaries_gt,  "GeoThought val",  LATENT_EUCLID_V4)
_print_table(summaries_g3k, "Geometry3K test")

# convenience aliases for downstream cells
summary_vl_direct_gt      = next((s for s in summaries_gt  if "direct"    in s["model"] and "VL" in s["model"]), None)
summary_vl_direct_g3k     = next((s for s in summaries_g3k if "direct"    in s["model"] and "VL" in s["model"]), None)
summary_vl_reasoning_gt   = next((s for s in summaries_gt  if "reasoning" in s["model"] and "VL" in s["model"]), None)
summary_vl_reasoning_g3k  = next((s for s in summaries_g3k if "reasoning" in s["model"] and "VL" in s["model"]), None)
summary_base_text_gt      = next((s for s in summaries_gt  if "text-only" in s["model"]), None)
summary_base_text_g3k     = next((s for s in summaries_g3k if "text-only" in s["model"]), None)
summary_base_reasoning_gt = next((s for s in summaries_gt  if "reasoning" in s["model"] and "Base" in s["model"]), None)
summary_base_reasoning_g3k= next((s for s in summaries_g3k if "reasoning" in s["model"] and "Base" in s["model"]), None)


In [ ]:
# ── Cell 14: Accuracy comparison — both datasets ─────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

_MODEL_DEFS = [
    ("VL-4B\ndirect",         "#4C72B0", summary_vl_direct_gt,      summary_vl_direct_g3k),
    ("Base-4B\ntext-only",    "#C44E52", summary_base_text_gt,      summary_base_text_g3k),
    ("Base-4B\nreasoning",    "#8172B2", summary_base_reasoning_gt, summary_base_reasoning_g3k),
    ("VL-4B\nreasoning",      "#55A868", summary_vl_reasoning_gt,   summary_vl_reasoning_g3k),
    ("LatentEuclid\nv4 SOTA", "#DD8452", LATENT_EUCLID_V4,          None),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (ds_label, ds_idx) in zip(axes, [("GeoThought val", 2), ("Geometry3K test", 3)]):
    labels, accs, colors = [], [], []
    for lbl, col, *summaries in _MODEL_DEFS:
        s = summaries[ds_idx - 2]
        if s is None: continue
        labels.append(lbl); accs.append(s["accuracy"]); colors.append(col)
    bars = ax.bar(labels, accs, color=colors, edgecolor="white", linewidth=0.8)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{acc:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_ylim(0, 75)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"Accuracy — {ds_label}")
    ax.axhline(25, color="gray", linestyle="--", linewidth=0.8, label="Random (25%)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("GeoThought Baseline Comparison", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "baseline_accuracy_comparison.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")


In [ ]:
# ── Cell 15: Error breakdown — both datasets ─────────────────────────────────
def classify_errors(results):
    empty, wrong = 0, 0
    for r in results:
        if r["is_correct"]: continue
        if not r["pred_norm"] or r["pred_norm"] in ("", "none", "nan"): empty += 1
        else: wrong += 1
    return empty, wrong

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (ds_label, model_data) in zip(axes, [
    ("GeoThought val", [
        ("VL direct",  results_vl_direct_gt,      summary_vl_direct_gt),
        ("Base text",  results_base_text_gt,      summary_base_text_gt),
        ("Base CoT",   results_base_reasoning_gt, summary_base_reasoning_gt),
        ("VL CoT",     results_vl_reasoning_gt,   summary_vl_reasoning_gt),
    ]),
    ("Geometry3K test", [
        ("VL direct",  results_vl_direct_g3k,      summary_vl_direct_g3k),
        ("Base text",  results_base_text_g3k,      summary_base_text_g3k),
        ("Base CoT",   results_base_reasoning_g3k, summary_base_reasoning_g3k),
        ("VL CoT",     results_vl_reasoning_g3k,   summary_vl_reasoning_g3k),
    ]),
]):
    model_data = [(n, r, s) for n, r, s in model_data if r and s is not None]
    if not model_data: ax.set_title(f"{ds_label} — no results"); continue
    x      = np.arange(len(model_data))
    labels = [d[0] for d in model_data]
    corr   = [d[2]["correct"] for d in model_data]
    empties, wrongs = zip(*[classify_errors(d[1]) for d in model_data])
    width = 0.25
    ax.bar(x - width, corr,    width, label="Correct",             color="#55A868")
    ax.bar(x,         wrongs,  width, label="Wrong answer",        color="#C44E52")
    ax.bar(x + width, empties, width, label="Empty / unparseable", color="#8c8c8c")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Sample count"); ax.set_title(f"Error breakdown — {ds_label}")
    ax.legend(); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
err_fig_path = os.path.join(RESULTS_DIR, "baseline_error_breakdown.png")
plt.savefig(err_fig_path, dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Cell 16: Cross-model overlap (GeoThought val) ────────────────────────────
import random as _rand

for ds_label, items, combined, result_map in [
    ("GeoThought val", geothought_items, combined_gt, {
        "VL direct":  (results_vl_direct_gt,      "qwen_vl_direct"),
        "Base text":  (results_base_text_gt,      "qwen_base_text"),
        "Base CoT":   (results_base_reasoning_gt, "qwen_base_reasoning"),
        "VL CoT":     (results_vl_reasoning_gt,   "qwen_vl_reasoning"),
    }),
    ("Geometry3K test", geometry3k_items, combined_g3k, {
        "VL direct":  (results_vl_direct_g3k,      "qwen_vl_direct"),
        "Base text":  (results_base_text_g3k,      "qwen_base_text"),
        "Base CoT":   (results_base_reasoning_g3k, "qwen_base_reasoning"),
        "VL CoT":     (results_vl_reasoning_g3k,   "qwen_vl_reasoning"),
    }),
]:
    avail = {k: v for k, v in result_map.items() if v[0]}
    if len(avail) < 2:
        print(f"{ds_label}: need ≥ 2 baselines, skipping."); continue

    all_img = {it["image_path"] for it in items}
    correct_sets = {k: {r["image"] for r in v[0] if r["is_correct"]} for k, v in avail.items()}
    names = list(correct_sets.keys())
    sets  = [correct_sets[n] for n in names]

    print(f"\n{'='*50}  {ds_label}")
    all_correct = sets[0]
    for s in sets[1:]: all_correct = all_correct & s
    print(f"  All {len(names)} correct: {len(all_correct)}")
    for n, s in zip(names, sets):
        others = [correct_sets[m] for m in names if m != n]
        unique = s.copy()
        for o in others: unique -= o
        print(f"  {n} only: {len(unique)}")
    all_wrong = all_img
    for s in sets: all_wrong -= s
    print(f"  All wrong: {len(all_wrong)}")

    combined_by_img = {c["image"]: c for c in combined}
    _rand.seed(0)
    for label, (lst, combined_key) in avail.items():
        others = [correct_sets[m] for m in names if m != label]
        unique = correct_sets[label].copy()
        for o in others: unique -= o
        if not unique: continue
        sample = _rand.sample(sorted(unique), min(2, len(unique)))
        print(f"\n  --- {label} only ({len(unique)}) ---")
        for img in sample:
            c = combined_by_img.get(img, {})
            print(f"    Q: {c.get('question','?')[:70]}")
            print(f"    GT: {c.get('gt_raw','?')}  |  Pred: {(c.get(combined_key) or {}).get('pred_raw','?')}")
